# 18 — MLP Classifier (Multi-Layer Perceptron)

**Algorithm:** MLPClassifier — feedforward neural network (256→128 hidden units)  
**Features:** TF-IDF (50K unigrams + bigrams)  
sklearn's MLP is a fully connected neural network trained with backpropagation. The `loss_curve_` attribute gives a direct epoch-by-epoch convergence trace.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve, plot_learning_curve)
from sklearn.metrics import accuracy_score
BASE_DIR, FIGURES_DIR = setup_paths()

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## Model Training

In [ ]:
from sklearn.neural_network import MLPClassifier
MODEL_NAME = 'MLP Classifier'
model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    max_iter=100,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=False
)
t0 = time.time()
model.fit(X_tr, y_train)
train_time = time.time() - t0
print(f'Training completed in {train_time:.2f}s')
print(f'Iterations run : {model.n_iter_}')
print(f'Best val score : {model.best_validation_score_:.4f}')

In [ ]:
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## Training Convergence: Loss & Validation Score per Epoch

In [ ]:
loss_curve = model.loss_curve_
val_scores  = model.validation_scores_
epochs = np.arange(1, len(loss_curve) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, loss_curve, color='#e74c3c', lw=2, label='Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('MLP — Training Loss Convergence')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

# Validation accuracy
axes[1].plot(np.arange(1, len(val_scores)+1), val_scores,
             color='#3498db', lw=2, label='Validation Accuracy')
best_ep = np.argmax(val_scores) + 1
axes[1].axvline(best_ep, color='green', linestyle='--', lw=1.5,
               label=f'Best epoch = {best_ep}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('MLP — Validation Accuracy per Epoch')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('MLP Classifier — Training Convergence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'mlp_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()

## Hidden Layer Size Comparison

In [ ]:
configs = [
    ((64,),      '64'),
    ((128,),     '128'),
    ((256,),     '256'),
    ((128, 64),  '128→64'),
    ((256, 128), '256→128'),
    ((256, 128, 64), '256→128→64'),
]
accs = []
for layers, label in configs:
    m = MLPClassifier(hidden_layer_sizes=layers, max_iter=50, random_state=42,
                      early_stopping=True, verbose=False)
    m.fit(X_tr, y_train)
    accs.append(accuracy_score(y_test, m.predict(X_te)))
    print(f'  {label}: {accs[-1]:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar([c[1] for c in configs], accs, color='#9b59b6', alpha=0.85)
for bar, v in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Test Accuracy')
ax.set_title('MLP — Architecture Comparison')
ax.set_ylim(min(accs) - 0.02, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'mlp_architecture_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Learning Curve

In [ ]:
plot_learning_curve(model, X_tr, y_train, X_te, y_test, MODEL_NAME, FIGURES_DIR)